# 03 — Equilibrium and model mechanics

This notebook connects the game-theory definitions in the handbook to the implementation. First validate the small synthetic game; then inspect the real-study allocations. The cells call tested functions—they do not reproduce solver logic.

## What to be able to explain at a defense

- A player is one driver; a strategy is one eligible route.
- A payoff is expected route income after fares and operating costs.
- A Nash equilibrium is a profile where no one driver improves by switching alone.
- The social optimum minimizes the study's combined passenger-wait and income-equity objective.

Read the assumptions in `src/config.py` and the payoff functions in `src/payoffs.py` before treating the result as a prediction.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
os.chdir(root)

from src.analysis import run_full_analysis, validate_synthetic_example
from src.config import DEFAULT_ALPHA, DEFAULT_DRIVER_COUNT, DEFAULT_ROUTE_INFO, MODEL_ROUTES, ROUTES
from src.data_parser import build_boarding_counts
from src.demand import build_demand_tensor, load_field_sheets
from src.equilibrium import find_equilibrium, find_equilibrium_multistart
from src.lptrp import load_lptrp_profile
from src.paths import ProjectPaths

paths = ProjectPaths.discover()

In [ ]:
# A known 3-driver case provides a fast solver sanity check.
validation = validate_synthetic_example()
pd.Series(validation, name='synthetic validation')

In [ ]:
if not paths.processed_data.exists():
    build_boarding_counts(paths.raw_data, paths.processed_data)

field_data = load_field_sheets(paths.processed_data)
demand = build_demand_tensor(field_data)
n_drivers = DEFAULT_DRIVER_COUNT
lptrp_profile = load_lptrp_profile(paths.lptrp_profile, n_drivers)
print(f'Demand tensor: {demand.shape}; drivers: {n_drivers}')

## Nash equilibrium diagnostic

Best-response dynamics changes one driver at a time until a full iteration produces no profitable unilateral switch. The convergence log lets you see that stabilization rather than assume it.

In [ ]:
initial_profile = np.random.default_rng(42).integers(0, len(ROUTES), size=n_drivers)
nash_profile, convergence_log = find_equilibrium(initial_profile, demand, DEFAULT_ROUTE_INFO)

allocation = pd.DataFrame({
    'route': MODEL_ROUTES,
    'nash_drivers': np.bincount(nash_profile, minlength=len(ROUTES))[:len(MODEL_ROUTES)],
    'lptrp_drivers': np.bincount(lptrp_profile, minlength=len(ROUTES))[:len(MODEL_ROUTES)],
})
display(allocation)
pd.DataFrame(convergence_log, columns=['iteration', 'drivers_switched']).tail()

In [ ]:
# Multiple starts test whether the algorithm discovers materially different allocations.
multi_start_counts, unique_equilibria = find_equilibrium_multistart(
    demand, DEFAULT_ROUTE_INFO, n_drivers, n_starts=5, seed=42
)
pd.DataFrame(multi_start_counts, columns=ROUTES).loc[:, list(MODEL_ROUTES)]

## Full welfare comparison (deliberately opt-in)

The social-optimum search is more expensive than finding one equilibrium. Set `RUN_FULL_ANALYSIS` to `True` when you want to reproduce this comparison; use `04_results.ipynb` for the official complete pipeline.

In [ ]:
RUN_FULL_ANALYSIS = False

if RUN_FULL_ANALYSIS:
    results = run_full_analysis(
        demand, DEFAULT_ROUTE_INFO, n_drivers, lptrp_profile, alpha=DEFAULT_ALPHA
    )
    print(f"Price of Anarchy: {results['price_of_anarchy']:.3f}")
    print(f"LPTRP improvement ratio: {results['lptrp_improvement_ratio']:.3f}")
else:
    print('Set RUN_FULL_ANALYSIS = True to calculate the social optimum and welfare metrics.')